In [3]:

from zne_estimator import ZNEEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager


transpiler = generate_preset_pass_manager(optimization_level=2)

runtime_estimator = AerEstimator()
zne = ZNEEstimator(base_estimator=runtime_estimator)  # no factors at construction

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
observable = SparsePauliOp("Z" * circuit.num_qubits)

transpiled_circuit = transpiler.run(circuit)  # assume transpilation is done here


job = zne.run(
    [(transpiled_circuit, observable)],
    precision=0.01,
    noise_factors=[1, 3, 5],
)
result = job.result()
print(result)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>)), metadata={'zne': {'noise_factors': [1, 3, 5], 'extrapolator': 'linear', 'noisy_evs': array([1.00454394, 0.98284463, 0.99856294]), 'noisy_stds': array([0.01, 0.01, 0.01]), 'target_precision': 0.01}})], metadata={'zne': True, 'noise_factors': [1, 3, 5]})


In [12]:
from qiskit.circuit import Instruction
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(5)

def _make_folded_block(self, circuit: QuantumCircuit) -> Instruction:
    """Build a named instruction representing one (U^dagger U) fold."""
    block = QuantumCircuit(circuit.num_qubits, name="zne_fold")
    block.barrier()
    block.compose(circuit.inverse(), inplace=True)
    block.barrier()
    block.compose(circuit, inplace=True)
    
    return block.to_instruction()


def _fold_circuit(self, circuit: QuantumCircuit, noise_factor: float) -> QuantumCircuit:
    n = int(round(noise_factor))
    if n == 1:
        return circuit.copy()

    folded = circuit.copy_empty_like()
    folded.compose(circuit, inplace=True)

    fold_block = _make_folded_block("", circuit)
    for _ in range((n - 1) // 2):
        folded.append(fold_block, folded.qubits)

    folded.metadata = {"zne_folded": True, "zne_noise_factor": noise_factor}
    return folded

# Checking if the levels of optimization change the folded circuit
for op_level in range(4):
    transpiler = generate_preset_pass_manager(optimization_level=op_level, backend=backend)
    transpiled_2 = transpiler.run(circuit)
    folded = _fold_circuit("", transpiled_2, 3)

    transpiled_3 = transpiler.run(folded)
    print(f"Optimization level {op_level}:")
    print("Actual folded circuit:")
    print(folded)
    print("Transpiled folded circuit:")
    print(transpiled_3)

Optimization level 0:
Actual folded circuit:
global phase: π/2
               ┌─────────┐┌────┐┌─────────┐     ┌───────────┐
      q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├──■──┤0          ├
               └─────────┘└────┘└─────────┘┌─┴─┐│           │
      q_1 -> 1 ────────────────────────────┤ X ├┤1          ├
                                           └───┘│           │
ancilla_0 -> 2 ─────────────────────────────────┤2 zne_fold ├
                                                │           │
ancilla_1 -> 3 ─────────────────────────────────┤3          ├
                                                │           │
ancilla_2 -> 4 ─────────────────────────────────┤4          ├
                                                └───────────┘
Transpiled folded circuit:
global phase: π
         ┌─────────┐┌────┐┌─────────┐      ░      ┌──────────┐┌─────────┐»
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├──■───░───■──┤ Rz(-π/2) ├┤ Rz(π/2) ├»
         └─────────┘└────┘└─────────┘┌─┴─┐ ░ ┌─┴─┐└──────────┘